In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os, joblib
import numpy as np
import pandas as pd
import polars as pl

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.linear_model import RidgeCV

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.model_selection import train_test_split

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

# Read data
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def advanced_feature_engineering(data):
    """Create advanced features from raw data - STATELESS transformations only"""
    df = data.copy()
    
    # 1. Economic Features (E-series) - Aggregations across E features
    e_features = [f'E{i}' for i in range(1, 21)]
    available_e = [f for f in e_features if f in df.columns]
    
    if len(available_e) > 0:
        df['E_mean'] = df[available_e].mean(axis=1)
        df['E_std'] = df[available_e].std(axis=1)
        df['E_median'] = df[available_e].median(axis=1)
        df['E_max'] = df[available_e].max(axis=1)
        df['E_min'] = df[available_e].min(axis=1)
        df['E_range'] = df['E_max'] - df['E_min']
        df['E_skew'] = df[available_e].skew(axis=1)
        df['E_kurt'] = df[available_e].kurtosis(axis=1)
        df['E_coeff_var'] = df['E_std'] / (df['E_mean'].abs() + 1e-10)
        df['E_q25'] = df[available_e].quantile(0.25, axis=1)
        df['E_q75'] = df[available_e].quantile(0.75, axis=1)
        df['E_iqr'] = df['E_q75'] - df['E_q25']
    
    # 2. Interaction features between key variables
    if 'S1' in df.columns and 'S2' in df.columns:
        df['S1_S2_ratio'] = df['S1'] / (df['S2'].abs() + 1e-10)
        df['S1_S2_product'] = df['S1'] * df['S2']
        df['S1_S2_diff'] = df['S1'] - df['S2']
    
    if 'S5' in df.columns and 'S1' in df.columns:
        df['S5_S1_ratio'] = df['S5'] / (df['S1'].abs() + 1e-10)
        df['S5_S1_product'] = df['S5'] * df['S1']
    
    if 'P8' in df.columns and 'P9' in df.columns:
        df['P8_P9_ratio'] = df['P8'] / (df['P9'].abs() + 1e-10)
        df['P8_P9_product'] = df['P8'] * df['P9']
        df['P8_P9_diff'] = df['P8'] - df['P9']
    
    if 'P10' in df.columns and 'P12' in df.columns:
        df['P10_P12_ratio'] = df['P10'] / (df['P12'].abs() + 1e-10)
        df['P10_P12_product'] = df['P10'] * df['P12']
    
    if 'P13' in df.columns and 'P10' in df.columns:
        df['P13_P10_ratio'] = df['P13'] / (df['P10'].abs() + 1e-10)
    
    if 'I2' in df.columns and 'S1' in df.columns:
        df['I2_S1_ratio'] = df['I2'] / (df['S1'].abs() + 1e-10)
        df['I2_S1_product'] = df['I2'] * df['S1']
    
    # 3. Polynomial features for key indicators
    key_features = ['S1', 'S2', 'S5', 'P8', 'P9', 'P10', 'I2']
    for feat in key_features:
        if feat in df.columns:
            df[f'{feat}_squared'] = df[feat] ** 2
            df[f'{feat}_cubed'] = df[feat] ** 3
            df[f'{feat}_sqrt'] = np.sign(df[feat]) * np.sqrt(np.abs(df[feat]))
            df[f'{feat}_log'] = np.sign(df[feat]) * np.log1p(np.abs(df[feat]))
    
    # 4. Cross-group interactions
    if 'E_mean' in df.columns and 'S1' in df.columns:
        df['E_mean_S1_product'] = df['E_mean'] * df['S1']
        df['E_mean_S1_ratio'] = df['E_mean'] / (df['S1'].abs() + 1e-10)
    
    if 'E_mean' in df.columns and 'P9' in df.columns:
        df['E_mean_P9_product'] = df['E_mean'] * df['P9']
    
    # 5. Binning features (based on absolute values, not data-dependent)
    if 'S1' in df.columns:
        df['S1_sign'] = np.sign(df['S1'])
        df['S1_abs'] = np.abs(df['S1'])
    
    if 'S2' in df.columns:
        df['S2_sign'] = np.sign(df['S2'])
        df['S2_abs'] = np.abs(df['S2'])
    
    if 'P9' in df.columns:
        df['P9_sign'] = np.sign(df['P9'])
        df['P9_abs'] = np.abs(df['P9'])
    
    # 6. Economic feature subgroups
    e_early = [f'E{i}' for i in range(1, 8) if f'E{i}' in df.columns]
    e_mid = [f'E{i}' for i in range(8, 15) if f'E{i}' in df.columns]
    e_late = [f'E{i}' for i in range(15, 21) if f'E{i}' in df.columns]
    
    if len(e_early) > 0:
        df['E_early_mean'] = df[e_early].mean(axis=1)
    if len(e_mid) > 0:
        df['E_mid_mean'] = df[e_mid].mean(axis=1)
    if len(e_late) > 0:
        df['E_late_mean'] = df[e_late].mean(axis=1)
    
    if 'E_early_mean' in df.columns and 'E_late_mean' in df.columns:
        df['E_early_late_ratio'] = df['E_early_mean'] / (df['E_late_mean'].abs() + 1e-10)
    
    return df

def preprocessing(data, typ, scaler=None, feature_stats=None):
    """
    Enhanced preprocessing with feature engineering
    
    Args:
        data: Input dataframe
        typ: 'train' or 'inference'
        scaler: Fitted scaler (for inference)
        feature_stats: Dict with median values for imputation (for inference)
    
    Returns:
        Processed data, scaler, feature_stats
    """
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',
                     "S2", "P9", "S1", "S5", "I2", "P8", "P10", "P12", "P13"]
    
    # Create feature-engineered dataset
    data_enhanced = advanced_feature_engineering(data)
    
    if typ == "train":
        # Keep target
        if 'forward_returns' in data_enhanced.columns:
            data_enhanced = data_enhanced.rename(columns={'forward_returns': 'target'})
        
        # Drop rows with null targets first
        data_enhanced = data_enhanced.dropna(subset=['target'])
        
        # Separate features and target
        feature_cols = [col for col in data_enhanced.columns if col != 'target']
        
        # Calculate median for each feature (for imputation)
        feature_stats = {}
        for col in feature_cols:
            feature_stats[col] = data_enhanced[col].median()
            data_enhanced[col].fillna(feature_stats[col], inplace=True)
        
        # Scale features
        scaler = RobustScaler()
        data_enhanced[feature_cols] = scaler.fit_transform(data_enhanced[feature_cols])
        
        return data_enhanced, scaler, feature_stats
    
    else:  # inference
        # Get all columns that should exist (from training)
        if feature_stats is None:
            raise ValueError("feature_stats must be provided for inference")
        
        # Ensure all training features exist
        for col in feature_stats.keys():
            if col not in data_enhanced.columns:
                data_enhanced[col] = 0  # Add missing columns with default value
        
        # Select only the features that were in training
        feature_cols = list(feature_stats.keys())
        data_enhanced = data_enhanced[feature_cols].copy()
        
        # Fill nulls using training statistics
        for col in feature_cols:
            data_enhanced[col].fillna(feature_stats[col], inplace=True)
        
        # Scale features using fitted scaler
        if scaler is not None:
            data_enhanced[feature_cols] = scaler.transform(data_enhanced[feature_cols])
        
        return data_enhanced

# Apply preprocessing
train_processed, scaler, feature_stats = preprocessing(train, "train")

# Split data
train_split, val_split = train_test_split(
    train_processed, test_size=0.1, random_state=42, shuffle=False
)

train_x = train_split.drop(columns=["target"])
test_x = val_split.drop(columns=["target"])
train_y = train_split['target']
test_y = val_split['target']

print(f"Original features: 29")
print(f"Enhanced features: {train_x.shape[1]}")
print(f"Training samples: {len(train_x)}")
print(f"Validation samples: {len(test_x)}")
print(f"\nFeature columns saved for inference: {len(feature_stats)}")
print(f"\nFirst 20 features:")
print(train_x.columns.tolist()[:20])

# Example: How to use for inference
# test_data = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv')
# test_processed = preprocessing(test_data, "inference", scaler=scaler, feature_stats=feature_stats)
# predictions = model.predict(test_processed)

Original features: 29
Enhanced features: 163
Training samples: 8091
Validation samples: 899

Feature columns saved for inference: 163

First 20 features:
['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18']


In [3]:
train_processed

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,S1_sign,S1_abs,S2_sign,S2_abs,P9_sign,P9_abs,E_early_mean,E_mid_mean,E_late_mean,E_early_late_ratio
0,-1.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
1,-0.999778,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
2,-0.999555,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
3,-0.999333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
4,-0.999110,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,0.999110,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,-0.844039,-1.0,-0.386794,0.0,-0.215775,0.269567,0.251418,0.088100,0.659446
8986,0.999333,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,-0.778316,-1.0,-0.829757,0.0,-0.219402,0.275695,0.249998,-0.078867,1.235093
8987,0.999555,0.0,0.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,-0.679827,-1.0,-0.561751,0.0,-0.213962,0.284229,0.248579,-0.098418,1.355013
8988,0.999778,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,-0.625398,-1.0,-0.560297,0.0,-0.205802,0.293889,0.247159,-0.043767,1.103623


In [4]:
LGBM_R_parm = {'boosting_type': 'gbdt', 
               'colsample_bytree': 0.9484106149593443, 
               'learning_rate': 0.1988123373955639, 
               'max_bin': 77, 
               'max_depth': 10, 
               'metric': 'mape', 
               'min_child_samples': 81, 
               'min_data_in_leaf': 21, 
               'n_estimators': 5029, 
               'num_leaves': 42, 
               'objective': 'regression_l1', 
               'reg_alpha': 0.6355835028602363, 
               'reg_lambda': 3.109823217156622, 
               'subsample': 0.7300733288106989, 
               'verbosity': -1}

print(f'LGBMRegressor TRAINING...')
LGBM_R = LGBMRegressor(**LGBM_R_parm)
LGBM_R.fit(train_x, train_y, eval_set=[(test_x, test_y)])
joblib.dump(LGBM_R, 'LGBM_R.pkl')

LGBMRegressor TRAINING...


['LGBM_R.pkl']

In [5]:
LGBM_R_model = joblib.load('LGBM_R.pkl')

In [6]:
# Signal conversion parameters (like in working code)
SIGNAL_MULTIPLIER = 400.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

def convert_ret_to_signal(ret_value):
    """Convert raw prediction to signal in range [0.0, 2.0]"""
    return np.clip(ret_value * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

def predict(test: pl.DataFrame) -> float:
        test_df = test.to_pandas()
        
        # Preprocess - use 'inference' type to handle test data properly
        test_processed = preprocessing(test_df, 'inference', scaler, feature_stats)[[i for i in list(train_processed.columns) if i != 'target']]
        #print(test_processed)
        
        LGBM_R_y_pred = LGBM_R_model.predict(test_processed)
        signal = convert_ret_to_signal(LGBM_R_y_pred)
        #print(float(signal))
        
        return float(signal)
        
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))